# 🏪 Zava Agentic Fine-Tuning Lab — 05: Training Results & Evaluate

**In this notebook**, you'll work through sections 7 and 8 from the lab:
- **Section 7** — Explore the reward curve, token usage, and checkpoint comparison from a completed RFT run
- **Section 8** — Evaluate the fine-tuned model head-to-head against base o4-mini

| What you'll do | Time |
|----------------|------|
| Plot reward curve + training metrics | 5 min |
| Compare checkpoints | 3 min |
| Run head-to-head evaluation | 10 min |

> **Prerequisite**: Complete `04-build-data-submit-job.ipynb` first.

---
## Setup — Reconnect and Load Agent Infrastructure

In [ ]:
import json, os, re, time, textwrap, csv
import requests
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

# connection for your foundry project
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential = DefaultAzureCredential()
)
my_client = project_client.get_openai_client(api_key=os.environ["API_KEY"])

# connection for the o4-mini model we already trained for you!
proxy_client = AzureOpenAI(
    azure_endpoint="", #will be provided on skillable
    api_key="", # will be provided on skillable
    api_version="2024-12-01-preview",
)
print("✅ Connected to Microsoft Foundry")

In [ ]:
# === Tool endpoint (pre-deployed Azure Function) ===
TOOL_URL = "https://zava-rft-tools.azurewebsites.net"

# The system prompt the agent uses
SYSTEM_PROMPT = """You are Zava's return resolution engine. Call get_order to look up order details, then apply the return policy to compute the resolution.

POLICY: Standard=30d/15d(electronics), Gold=45d/30d, Platinum=60d/45d. Electronics restocking: Std=15%, Gold=7.5%, Plat=0%. Defective=0%. Sale=final sale (defective sale→store credit). Late delivery(>2d)=$10 credit +15d extension. Lost=replacement/refund. Pending=cancellable. Opened personal care=deny unless defective.

Respond with your resolution including: action, amounts, and policy reasoning."""

# Tool definition (same schema the model sees)
TOOLS = [
    {"type": "function", "function": {
        "name": "get_order",
        "description": "Look up order details including items, prices, dates, loyalty tier, and delivery status.",
        "parameters": {"type": "object", "properties": {
            "order_id": {"type": "string", "description": "The order ID (e.g., ORD-003)"}
        }, "required": ["order_id"]}
    }}
]


def call_tool(name, args):
    """Call the Zava tool endpoint and return the result."""
    url = f"{TOOL_URL}/tool/{name}"
    payload = {"arguments": json.dumps(args), "call_id": "c", "id": "f", "trace_id": "t"}
    r = requests.post(url, json=payload, timeout=30)
    return r.json().get("output", json.dumps(r.json()))


def run_agent(user_message, client, model="o4-mini", verbose=True):
    """Run the full agent loop: model → tool call → model → response."""
    messages = [
        {"role": "developer", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    tool_calls_made = []

    for turn in range(8):  # max 8 turns to prevent infinite loops
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, max_completion_tokens=8192
        )
        msg = resp.choices[0].message

        # Build assistant message for conversation history
        assistant_msg = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
            tool_calls_made.extend(msg.tool_calls)
        messages.append(assistant_msg)

        # If no tool calls, we're done
        if not msg.tool_calls:
            if verbose and msg.content:
                print(f"\n📋 Agent Response:\n{textwrap.fill(msg.content, width=80)}")
            return msg.content or "", tool_calls_made

        # Execute tool calls
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"  🔧 Calling {tc.function.name}({args})")
            result = call_tool(tc.function.name, args)
            if verbose:
                # Show a preview of the tool result
                preview = result[:200] + "..." if len(result) > 200 else result
                print(f"  📦 Result: {preview}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "", tool_calls_made


def python_grader(output_text, output_tools, expected_resolution):
    """Score a model response against the expected resolution.
    Returns 0.0 to 1.0 — same logic used during RFT training."""
    if not expected_resolution:
        return 0.5

    score = 0.0
    exp_lower = expected_resolution.lower()
    out_lower = (output_text or "").lower()

    # Action correctness (0.4)
    actions = {
        "refund": ["refund"],
        "denied": ["denied", "deny", "not eligible", "cannot", "expired"],
        "store credit": ["store credit", "store_credit"],
        "replacement": ["replacement", "replace"],
        "exchange": ["exchange", "swap"],
        "cancel": ["cancel", "cancellation"],
    }
    for action, keywords in actions.items():
        if any(k in exp_lower for k in keywords):
            if any(k in out_lower for k in keywords):
                score += 0.4
            break

    # Amount correctness (0.3)
    exp_amounts = re.findall(r'\$(\d+\.\d{2})', expected_resolution)
    if exp_amounts:
        out_amounts = re.findall(r'\$(\d+\.\d{2})', output_text or "")
        hits = sum(1 for a in exp_amounts if a in out_amounts)
        score += 0.3 * (hits / len(exp_amounts))
    else:
        score += 0.15

    # Policy reasoning (0.2)
    policy_terms = ["window", "restocking", "defective", "sale", "platinum", "gold",
                    "standard", "late", "shipping credit", "personal care", "eligible"]
    exp_terms = [t for t in policy_terms if t in exp_lower]
    if exp_terms:
        hits = sum(1 for t in exp_terms if t in out_lower)
        score += 0.2 * (hits / len(exp_terms))

    # Tool usage bonus (0.1)
    if output_tools:
        tool_names = [t.function.name if hasattr(t, 'function') else t.get("function", {}).get("name", "") for t in output_tools]
        if "get_order" in tool_names:
            score += 0.1

    return round(min(score, 1.0), 3)


In [ ]:
# Load validation scenarios — use cached ground truth if available (same as 03-baseline-grader.ipynb)
import os, random
random.seed(42)

GT_FILE = "data/rft_v7_val_gt.jsonl"

if os.path.exists(GT_FILE):
    with open(GT_FILE) as f:
        val_scenarios = [json.loads(line) for line in f]
    print(f"✅ Loaded {len(val_scenarios)} scenarios with cached ground truth from {GT_FILE}")
else:
    with open("data/rft_v7_val.jsonl") as f:
        val_scenarios = [json.loads(line) for line in f]
    print(f"⚠️  Ground truth file not found. Run 03-baseline-grader.ipynb first to generate it.")
    print(f"   Loaded {len(val_scenarios)} scenarios (expected_resolution will be empty → scores will be 0.5)")

eval_scenarios = random.sample(val_scenarios, min(8, len(val_scenarios)))

print(f"\nEvaluating base o4-mini on {len(eval_scenarios)} scenarios...\n")
base_scores = []

for i, ex in enumerate(eval_scenarios):
    msg = ex["messages"][-1]["content"]
    expected = ex.get("expected_resolution", "")

    output, tools = run_agent(msg, my_client, model="o4-mini", verbose=False)
    score = python_grader(output, tools, expected)
    base_scores.append(score)

    status = "✅" if score >= 0.9 else ("⚠️" if score >= 0.5 else "❌")
    print(f"  [{i+1:2d}] {score:.3f} {status}  {msg}")

base_avg = sum(base_scores) / len(base_scores)
base_p90 = sum(1 for s in base_scores if s >= 0.9) / len(base_scores)
base_p80 = sum(1 for s in base_scores if s >= 0.8) / len(base_scores)

print(f"\n{'='*50}")
print(f"  BASE o4-mini RESULTS")
print(f"  Average score: {base_avg:.1%}")
print(f"  Pass@0.9 (strict): {base_p90:.0%}")
print(f"  Pass@0.8 (good): {base_p80:.0%}")
print(f"{'='*50}")

---
## 7. Explore Training Results (10 min)

While your job queues, let's look at results from a completed run.
We pre-ran the same experiment — here's what happened during training.

In [ ]:
import csv
import matplotlib.pyplot as plt

# Load pre-run training metrics
with open("results/training_metrics.csv") as f:
    metrics = list(csv.DictReader(f))

steps = [int(r["step"]) for r in metrics]
train_rewards = [float(r["train_mean_reward"]) for r in metrics]
valid_rewards = [float(r["full_valid_mean_reward"]) if r.get("full_valid_mean_reward") else None for r in metrics]
comp_tokens = [float(r["completion_tokens_mean"]) for r in metrics]

# Plot reward curve
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Reward trajectory
ax = axes[0][0]
ax.plot(steps, train_rewards, "b-o", label="Train reward", markersize=4)
valid_steps = [s for s, v in zip(steps, valid_rewards) if v is not None]
valid_vals = [v for v in valid_rewards if v is not None]
ax.plot(valid_steps, valid_vals, "r-s", label="Validation reward", markersize=6)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Training Step")
ax.set_ylabel("Mean Reward")
ax.set_title("📈 Reward Trajectory — Is the model learning?")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Completion tokens (response length)
ax = axes[0][1]
ax.plot(steps, comp_tokens, "g-o", markersize=4)
ax.set_xlabel("Training Step")
ax.set_ylabel("Tokens")
ax.set_title("📝 Completion Tokens — Response length over training")
ax.grid(True, alpha=0.3)

# 3. Reasoning tokens (how hard the model is thinking)
reason_tokens = [float(r.get("reasoning_tokens_mean", 0) or 0) for r in metrics]
ax = axes[1][0]
ax.plot(steps, reason_tokens, "m-o", markersize=4)
ax.set_xlabel("Training Step")
ax.set_ylabel("Tokens")
ax.set_title("🧠 Reasoning Tokens — How hard is the model thinking?")
ax.grid(True, alpha=0.3)

# 4. Tool call errors (is the model making valid tool calls?)
tool_errors = [float(r.get("train_error_count_get_order", 0) or 0) * 100 for r in metrics]
ax = axes[1][1]
ax.plot(steps, tool_errors, "r-o", markersize=4)
ax.set_xlabel("Training Step")
ax.set_ylabel("Error Rate (%)")
ax.set_title("🔧 Tool Call Errors — Drops as model learns")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/training_dashboard.png", dpi=150)
plt.show()

In [ ]:
# Key takeaways from the training metrics
print("📊 What the metrics tell us:\n")
print(f"  Reward:    {train_rewards[0]:.3f} → {train_rewards[-1]:.3f}  (model learned to score higher)")
print(f"  Comp tokens: {comp_tokens[0]:.0f} → {comp_tokens[-1]:.0f}    (responses got more detailed)")
print(f"  Reasoning: {reason_tokens[0]:.0f} → {reason_tokens[-1]:.0f}    (model is 'thinking harder')")
print(f"  Tool errors: {tool_errors[0]:.1f}% → {tool_errors[-1]:.1f}%  (learned to make valid tool calls)")
print()
print("💡 Watch for these during training:")
print("   ✅ Reward increasing = model is learning")
print("   ✅ Tool errors decreasing = model making better tool calls")
print("   ⚠️  Completion tokens growing fast = possible verbosity bloat")
print("   ⚠️  Reasoning tokens doubling = more inference cost per request")

### Checkpoint comparison

RFT saves checkpoints periodically. The best checkpoint isn't always the last one!
Always evaluate multiple checkpoints with your actual task metric.

In [ ]:
# Pre-computed checkpoint results
checkpoints = {
    "Base o4-mini": {"avg": base_avg, "p90": base_p90, "p80": base_p80},
    "Step 95 (peak train reward)": {"avg": .937, "p90": 0.88, "p80": 0.88},
    "Step 115 (best eval)": {"avg": 0.96, "p90": 0.88, "p80": 100},
}

print(f"{'Model':<35} {'Avg Score':>10} {'P@0.9':>8} {'P@0.8':>8}")
print(f"{'-'*35} {'-'*10} {'-'*8} {'-'*8}")
for name, r in checkpoints.items():
    print(f"{name:<35} {r['avg']:>9.1%} {r['p90']:>7.0%} {r['p80']:>7.0%}")

---
## 8. Evaluate the Fine-Tuned Model (10 min)

The pre-deployed fine-tuned model is available as `o4-mini-2025`.
Let's run it on the same scenarios and compare head-to-head.

In [ ]:
# Evaluate fine-tuned model on the same scenarios
# (Uses the deployment name of the pre-deployed RFT model)


RFT_MODEL = "o4-mini-2025"  # <-- pre-deployed by facilitator

print(f"Evaluating {RFT_MODEL} on {len(eval_scenarios)} scenarios...\n")
rft_scores = []

for i, ex in enumerate(eval_scenarios):
    msg = ex["messages"][-1]["content"]
    expected = ex.get("expected_resolution", "")

    output, tools = run_agent(msg, proxy_client, model=RFT_MODEL, verbose=False)
    score = python_grader(output, tools, expected)
    rft_scores.append(score)

    # Show comparison with base
    base_sc = base_scores[i]
    delta = score - base_sc
    arrow = "📈" if delta > 0.05 else ("📉" if delta < -0.05 else "➡️")
    print(f"  [{i+1:2d}] base={base_sc:.3f} → rft={score:.3f} ({delta:+.3f}) {arrow}  {msg[:45]}")

rft_avg = sum(rft_scores) / len(rft_scores)
rft_p90 = sum(1 for s in rft_scores if s >= 0.9) / len(rft_scores)
rft_p80 = sum(1 for s in rft_scores if s >= 0.8) / len(rft_scores)

In [ ]:
# Head-to-head summary
print(f"\n{'='*60}")
print(f"  HEAD-TO-HEAD COMPARISON")
print(f"{'='*60}")
print(f"  {'Metric':<20} {'Base o4-mini':>15} {'RFT Model':>15} {'Delta':>10}")
print(f"  {'-'*20} {'-'*15} {'-'*15} {'-'*10}")
print(f"  {'Average score':<20} {base_avg:>14.1%} {rft_avg:>14.1%} {rft_avg-base_avg:>+9.1%}")
print(f"  {'Pass@0.9 (strict)':<20} {base_p90:>14.0%} {rft_p90:>14.0%} {rft_p90-base_p90:>+9.0%}")
print(f"  {'Pass@0.8 (good)':<20} {base_p80:>14.0%} {rft_p80:>14.0%} {rft_p80-base_p80:>+9.0%}")
print(f"{'='*60}")

# Visualize
improved = sum(1 for b, r in zip(base_scores, rft_scores) if r > b + 0.05)
same = sum(1 for b, r in zip(base_scores, rft_scores) if abs(r - b) <= 0.05)
worse = sum(1 for b, r in zip(base_scores, rft_scores) if r < b - 0.05)
print(f"\n  Improved: {improved}/{len(base_scores)} scenarios")
print(f"  Same:     {same}/{len(base_scores)} scenarios")
print(f"  Worse:    {worse}/{len(base_scores)} scenarios")

---
## 💡 Key Takeaways

- **RFT improved average score from ~79% to ~96%%** — a significant gain on policy-following accuracy
- **The reward curve confirms learning** — train and validation reward both rise, tool call errors fall
- **Step 115 beats Step 95 despite 25% lower training reward (0.67 vs 0.88) — always evaluate checkpoints on full held-out data, not just training metrics.
- **P@0.8 jumped from 62% → 100%%** — the model now passes the "good" bar on 10 out of 10 scenarios
- **Reasoning tokens and completion length grew** — improved accuracy comes with a small inference cost increase

**Next → Open `06-wrap-up.ipynb` to review what you learned and explore next steps.**